# 05 — Modelo Híbrido (CNN + MLP)
### GalaxyNet — Classificador Morfológico de Galáxias

Modelo de fusão tardia (*late fusion*) que combina:
- **Branch CNN**: imagens $(64 \times 64 \times 3)$ nas bandas $g$, $r$, $i$ — extrai features espaciais
- **Branch MLP**: 15 features tabulares fotométricas — captura propriedades globais

As representações são concatenadas e processadas por uma cabeça de classificação densa.
Inclui Data Augmentation, Class Weights e visualização **Grad-CAM** para interpretabilidade.

In [ ]:
import sys
import os
import pickle
import warnings
warnings.filterwarnings('ignore')

def _find_src_dir():
    current = os.path.abspath(os.getcwd())
    for _ in range(5):
        candidate = os.path.join(current, 'src')
        if os.path.isdir(candidate):
            return candidate
        current = os.path.dirname(current)
    raise RuntimeError("Diretório src/ não encontrado.")

src_path = _find_src_dir()
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.utils import to_categorical

from preprocessing import load_preprocessed_images, load_preprocessed_tabular
from models import create_hybrid_model
from evaluation import evaluate_galaxy_classifier
from visualization import compute_gradcam_hybrid, plot_gradcam_grid

# Reprodutibilidade
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Estilo dos gráficos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

# Caminhos
_project_root = os.path.dirname(src_path)
PROCESSED_DIR = os.path.join(_project_root, 'data', 'processed')
MODELS_DIR    = os.path.join(_project_root, 'models')
REPORTS_DIR   = os.path.join(_project_root, 'reports', 'figures')
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

print(f"TensorFlow    : {tf.__version__}")
print(f"PROCESSED_DIR : {PROCESSED_DIR}")
print(f"MODELS_DIR    : {MODELS_DIR}")

---
## 1. Carregamento dos Dados

Carrega os dados tabulares (15 features normalizadas) e as imagens pré-processadas.
Alinha ambos por `objid` para garantir correspondência correta.

In [ ]:
# Carregar dados tabulares
X_tabular, y_labels_tab, tab_objids, scaler = load_preprocessed_tabular(PROCESSED_DIR)

# Carregar imagens pré-processadas
X_images, img_objids = load_preprocessed_images(PROCESSED_DIR)

# Alinhar por objid (interseção)
common_objids = np.intersect1d(tab_objids, img_objids)
tab_idx = np.array([np.where(tab_objids == oid)[0][0] for oid in common_objids])
img_idx = np.array([np.where(img_objids == oid)[0][0] for oid in common_objids])

X_tabular = X_tabular[tab_idx]
X_images = X_images[img_idx]
y_labels = y_labels_tab[tab_idx]
objids = common_objids

print(f"X_tabular       : {X_tabular.shape}  dtype={X_tabular.dtype}")
print(f"X_images        : {X_images.shape}  dtype={X_images.dtype}")
print(f"y_labels        : {y_labels.shape}")
print(f"Imagens [0,1]   : min={X_images.min():.4f}, max={X_images.max():.4f}")

print("\nDistribuição de classes:")
unique, counts = np.unique(y_labels, return_counts=True)
for cls, n in zip(unique, counts):
    print(f"  {cls:<12}: {n:>4} ({100*n/len(y_labels):.1f}%)")

---
## 2. Encoding de Labels e Split Train/Val/Test

Carrega o `LabelEncoder` do notebook 03 para manter consistência entre modelos.
Split estratificado 70/15/15 aplicado simultaneamente aos dados tabulares e imagens.

In [ ]:
# Carregar LabelEncoder do notebook 03 para consistência
le_path = os.path.join(MODELS_DIR, 'label_encoder.pkl')
if os.path.exists(le_path):
    with open(le_path, 'rb') as f:
        label_encoder = pickle.load(f)
    print(f"LabelEncoder carregado de: {le_path}")
else:
    label_encoder = LabelEncoder()
    label_encoder.fit(y_labels)
    print("LabelEncoder ajustado localmente (label_encoder.pkl não encontrado)")

y_encoded = label_encoder.transform(y_labels)
y_one_hot = to_categorical(y_encoded)

num_classes = y_one_hot.shape[1]
print(f"Classes: {list(label_encoder.classes_)} → {list(range(num_classes))}")

# Split estratificado: 70% train, 15% val, 15% test
# Usar índices para manter tabulares e imagens alinhados
indices = np.arange(len(y_labels))
idx_train, idx_temp = train_test_split(
    indices, test_size=0.30, random_state=SEED, stratify=y_encoded
)
idx_val, idx_test = train_test_split(
    idx_temp, test_size=0.50, random_state=SEED, stratify=y_encoded[idx_temp]
)

X_tab_train, X_tab_val, X_tab_test = X_tabular[idx_train], X_tabular[idx_val], X_tabular[idx_test]
X_img_train, X_img_val, X_img_test = X_images[idx_train], X_images[idx_val], X_images[idx_test]
y_train, y_val, y_test = y_one_hot[idx_train], y_one_hot[idx_val], y_one_hot[idx_test]
objids_train, objids_val, objids_test = objids[idx_train], objids[idx_val], objids[idx_test]

print(f"\nTrain : {len(idx_train)} amostras")
print(f"Val   : {len(idx_val)} amostras")
print(f"Test  : {len(idx_test)} amostras")
print(f"  Tabular shape: train={X_tab_train.shape}, val={X_tab_val.shape}, test={X_tab_test.shape}")
print(f"  Image shape  : train={X_img_train.shape}, val={X_img_val.shape}, test={X_img_test.shape}")

for name, y_split in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    labels = label_encoder.inverse_transform(np.argmax(y_split, axis=1))
    unique, counts = np.unique(labels, return_counts=True)
    dist = ', '.join(f"{c}:{n}" for c, n in zip(unique, counts))
    print(f"  {name}: {dist}")

---
## 3. Data Augmentation e Class Weights

Data augmentation é aplicada apenas nas imagens (branch CNN). As features tabulares
passam diretamente sem transformação. Um gerador customizado `hybrid_generator`
combina ambas as entradas para o modelo dual-input.

- **Rotação 360°**, flips horizontal/vertical, zoom 0.2, brightness [0.8, 1.2]
- `fill_mode='constant'` com `cval=0.0` (fundo = céu escuro)

In [ ]:
# Data augmentation — apenas para imagens
datagen = ImageDataGenerator(
    rotation_range=360,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.2,
    brightness_range=[0.8, 1.2],
    fill_mode='constant',
    cval=0.0,
)

def hybrid_generator(X_tab, X_img, y, datagen, batch_size=32):
    """
    Generator que aplica data augmentation nas imagens e retorna
    [tabular, images_augmented], labels para o modelo dual-input.
    """
    n_samples = len(y)
    while True:
        indices = np.random.permutation(n_samples)
        for start in range(0, n_samples, batch_size):
            end = min(start + batch_size, n_samples)
            batch_idx = indices[start:end]

            tab_batch = X_tab[batch_idx]
            img_batch = X_img[batch_idx]
            y_batch = y[batch_idx]

            # Aplicar augmentation em cada imagem do batch
            aug_imgs = np.zeros_like(img_batch)
            for k in range(len(img_batch)):
                aug_imgs[k] = datagen.random_transform(img_batch[k])

            yield [tab_batch, aug_imgs], y_batch

# Class weights
y_train_integers = np.argmax(y_train, axis=1)
class_weights_array = compute_class_weight(
    'balanced',
    classes=np.unique(y_train_integers),
    y=y_train_integers,
)
class_weight_dict = dict(enumerate(class_weights_array))

print("Class weights (balanced):")
for idx, weight in class_weight_dict.items():
    cls_name = label_encoder.inverse_transform([idx])[0]
    n_train = (y_train_integers == idx).sum()
    print(f"  {cls_name:<12} (idx={idx}): weight={weight:.3f}  (n_train={n_train})")

# Steps por época
BATCH_SIZE = 32
steps_per_epoch = max(1, len(y_train) // BATCH_SIZE)
print(f"\nBatch size: {BATCH_SIZE}, Steps/epoch: {steps_per_epoch}")

---
## 4. Criação do Modelo Híbrido

Arquitetura Functional API com dois inputs (tabular + imagem) → fusão por concatenação.

In [ ]:
tabular_shape = X_tab_train.shape[1]  # 15
image_shape = X_img_train.shape[1:]   # (64, 64, 3)

hybrid_model = create_hybrid_model(tabular_shape, image_shape, num_classes)
hybrid_model.summary()

---
## 5. Treinamento

O treinamento usa o `hybrid_generator` para aplicar data augmentation nas imagens
enquanto passa as features tabulares diretamente. A validação usa dados originais.

- **EarlyStopping**: patience=15, restaura melhores pesos
- **ReduceLROnPlateau**: reduz LR pela metade se val_loss estagnar por 7 épocas

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=15,
        restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=7, min_lr=1e-7, verbose=1
    ),
]

train_gen = hybrid_generator(X_tab_train, X_img_train, y_train, datagen, BATCH_SIZE)

history = hybrid_model.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    validation_data=([X_tab_val, X_img_val], y_val),
    epochs=100,
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1,
)

print(f"\nTreinamento finalizado em {len(history.history['loss'])} épocas.")

---
## 6. Curvas de Treinamento (Loss e Accuracy)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Loss — Hybrid (CNN + MLP)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (Categorical Crossentropy)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history.history['accuracy'], label='Train Accuracy')
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[1].set_title('Accuracy — Hybrid (CNN + MLP)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'fig20_hybrid_training_curves.png'),
            bbox_inches='tight')
plt.show()
print('Figura salva: fig20_hybrid_training_curves.png')

---
## 7. Avaliação no Conjunto de Teste

In [ ]:
loss, accuracy = hybrid_model.evaluate([X_tab_test, X_img_test], y_test, verbose=0)
print(f"Hybrid Test Loss: {loss:.4f}, Test Accuracy: {accuracy:.4f}")

eval_results = evaluate_galaxy_classifier(
    model=hybrid_model,
    X_test_data=[X_tab_test, X_img_test],
    y_test_one_hot=y_test,
    label_encoder=label_encoder,
    model_type='Hybrid',
    save_dir=REPORTS_DIR,
)

---
## 8. Grad-CAM — Interpretabilidade

Grad-CAM (Gradient-weighted Class Activation Mapping) destaca as regiões da imagem
que mais contribuíram para a decisão do modelo. Aplicamos na última camada convolucional
do branch CNN do modelo híbrido.

Isso permite verificar se o modelo foca em características astrofísicas relevantes:
- **Elípticas**: distribuição de luz suave e centralizada
- **Espirais**: braços espirais e regiões de formação estelar
- **Irregulares**: estrutura assimétrica e clumpy

In [ ]:
# Identificar a última camada convolucional do branch CNN
conv_layers = [layer.name for layer in hybrid_model.layers
               if isinstance(layer, keras.layers.Conv2D)]
last_conv_layer = conv_layers[-1]
print(f"Última camada convolucional: {last_conv_layer}")

# Labels verdadeiros do conjunto de teste
y_test_labels = label_encoder.inverse_transform(np.argmax(y_test, axis=1))

# Gerar grid de Grad-CAM
fig = plot_gradcam_grid(
    model=hybrid_model,
    X_tabular=X_tab_test,
    X_images=X_img_test,
    y_true_labels=y_test_labels,
    layer_name=last_conv_layer,
    label_encoder=label_encoder,
    n_per_class=3,
    save_path=os.path.join(REPORTS_DIR, 'fig21_hybrid_gradcam.png'),
)
plt.show()

---
## 9. Salvar Modelo e Artefatos

In [ ]:
# Salvar modelo treinado
model_path = os.path.join(MODELS_DIR, 'hybrid_galaxy_classifier.h5')
hybrid_model.save(model_path)
print(f"Modelo salvo em: {model_path}")

# Salvar dados de teste para comparação posterior entre modelos
test_artifacts = {
    'X_tab_test': X_tab_test,
    'X_img_test': X_img_test,
    'y_test': y_test,
    'objids_test': objids_test,
}
test_path = os.path.join(MODELS_DIR, 'hybrid_test_data.pkl')
with open(test_path, 'wb') as f:
    pickle.dump(test_artifacts, f)
print(f"Dados de teste salvos em: {test_path}")

---
## 10. Resumo

In [ ]:
print("=" * 60)
print("RESUMO — MODELO HÍBRIDO (CNN + MLP)")
print("=" * 60)
print(f"Input tabular        : {tabular_shape} features")
print(f"Input imagem         : {image_shape}")
print(f"Classes              : {list(label_encoder.classes_)}")
print(f"Amostras (train/val/test): {len(idx_train)}/{len(idx_val)}/{len(idx_test)}")
print(f"Data augmentation    : rotation=360°, flips, zoom=0.2, brightness=[0.8,1.2]")
print(f"Épocas treinadas     : {len(history.history['loss'])}")
print(f"Test Loss            : {loss:.4f}")
print(f"Test Accuracy        : {accuracy:.4f}")
print()
print("Métricas Científicas:")
print(eval_results['scientific_metrics'].to_string(index=False))
print()
print("Artefatos salvos:")
print(f"  1. {model_path}")
print(f"  2. {test_path}")
print(f"  3. reports/figures/fig20_hybrid_training_curves.png")
print(f"  4. reports/figures/fig17_hybrid_confusion_matrix.png")
print(f"  5. reports/figures/fig21_hybrid_gradcam.png")